In [1]:
from ml.dataset import get_dataloaders
from ml.models import Conv1Layer
%load_ext autoreload
%autoreload 2

import logging
import torch
import torch.optim as optim
from torch import nn
from train import Trainer

logger = logging.getLogger("Main")
logging.basicConfig(
    filename="logger.log",
    filemode="w",
    level=logging.INFO,
)

In [2]:
torch.manual_seed(42)
torch.use_deterministic_algorithms(True) 

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

train_loader, val_loader, test_loader = get_dataloaders()

In [3]:
def run_experiment(
        model: nn.Module,
        model_name: str,
        lr: float = 0.001,
        max_epochs: int = 20,
        patience: int = 5,
) -> tuple[float, float, Trainer]:

    loss_fn  = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr = lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=2)

    trainer = Trainer(
        model = model,
        model_name = model_name,
        loss_fn = loss_fn,
        optimizer = optimizer,
        train_loader = train_loader,
        val_loader = val_loader,
        device = device,
        logger=logger,
        scheduler = scheduler,
    )

    val_loss, val_acc = trainer.train_and_validate(max_epochs = max_epochs, patience = patience)

    return val_loss, val_acc, trainer

In [4]:
# this structure is created so that the initialization is made
# lazily so it doesn't consume RAM unnecesarily
models = [
    ("Conv1_32", Conv1Layer, {"n1" : 32}),
    ("Conv1_64", Conv1Layer, {"n1" : 64}),
]

for model_name, model_cls, kwargs in models:
    model = model_cls(**kwargs)

    val_loss, val_acc, trainer = run_experiment(model = model, model_name = model_name)
    print(f"{model_name}: validation loss: {val_loss}, validation accuracy: {val_acc}")

    del model
    del trainer
    if torch.mps.is_available():
        torch.mps.empty_cache()


Conv1_32: validation loss: 2.123544402169709, validation accuracy: 0.596039603960396
Conv1_64: validation loss: 3.4893817764697688, validation accuracy: 0.5584158415841585
